# RAG

## Table of Contents
- [Environment setup](#environment-setup)
- [What is RAG?](#what-is-rag)
- [Calling an LLM: roles and temperature](#calling-an-llm-roles-and-temperature)
- [Tokens](#tokens)
- [Load only the user's text data](#load-only-the-users-text-data)
- [Embeddings and cosine similarity](#embeddings-and-cosine-similarity)
- [Chunking](#chunking)
- [Persistent vector search with Chroma](#persistent-vector-search-with-chroma)
- [Retrieval](#retrieval)
- [Augmentation and grounded generation](#augmentation-and-grounded-generation)
- [Run the complete pipeline](#run-the-complete-pipeline)
- [The honesty test](#the-honesty-test)
- [Lightweight evaluation and diagnostics](#lightweight-evaluation-and-diagnostics)

## Environment setup

~~~bash
# 1.Create a virtual environment
# macOS/Linux
python3 -m venv .venv
source .venv/bin/activate
# Windows PowerShell
python -m venv .venv
.venv\Scripts\Activate.ps1

# 2.Install dependencies
python -m pip install --upgrade pip
python -m pip install -r requirements.txt

# 3.Configure the model provide
# macOS/Linux
cp .env.example .env
# Windows PowerShell
Copy-Item .env.example .env
# Open .env and add your own API key:
OPENAI_API_KEY=your-key-here

# OR
# Use Ollama directly (no API key needed)
ollama serve
# Pull models
ollama pull llama3.2
ollama pull nomic-embed-text

# Start the notebook
jupyter notebook
~~~

Install Ollama, start it, and pull the models once before running the connection cells:

In [1]:
# Run this once if packages are not installed.
# %pip install -q openai chromadb python-dotenv tiktoken

from pathlib import Path
import hashlib
import os
import re

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
VECTOR_STORE_DIR = ROOT / ".rag_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

# Local-first configuration. Change RAG_PROVIDER to "openai" only when desired.
RAG_PROVIDER = os.getenv("RAG_PROVIDER", "ollama").strip().lower()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
OLLAMA_CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")

if RAG_PROVIDER == "ollama":
    BASE_URL = OLLAMA_BASE_URL
    API_KEY = "ollama"  # Ollama ignores this; the OpenAI SDK requires one.
    CHAT_MODEL = OLLAMA_CHAT_MODEL
    EMBEDDING_MODEL = OLLAMA_EMBEDDING_MODEL
elif RAG_PROVIDER == "openai":
    BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
    API_KEY = os.getenv("OPENAI_API_KEY")
    CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")
    EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
    if not API_KEY:
        raise RuntimeError("RAG_PROVIDER=openai requires OPENAI_API_KEY in the environment.")
else:
    raise ValueError("RAG_PROVIDER must be either 'ollama' or 'openai'.")

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
TEMPERATURE = 0.0
TOP_K = 4
MIN_RELEVANCE_SCORE = 0.20
CHUNK_SIZE_WORDS = 80
CHUNK_OVERLAP_WORDS = 20

print(f"Provider: {RAG_PROVIDER}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Data directory: {DATA_DIR}")
print(f"Vector store: {VECTOR_STORE_DIR}")

Provider: ollama
Chat model: llama3.2
Embedding model: nomic-embed-text
Data directory: C:\GitHub\RAG\data
Vector store: C:\GitHub\RAG\.rag_store


In [2]:
def check_connection():
    """Make an explicit connection check without hiding failures."""
    try:
        models = client.models.list()
        names = [model.id for model in models.data[:10]]
        print("Connection OK.")
        print("Visible models:", names if names else "(no model list returned)")
    except Exception as exc:
        if RAG_PROVIDER == "ollama":
            raise RuntimeError(
                "Could not reach Ollama. Run 'ollama serve', verify OLLAMA_BASE_URL, "
                f"and pull '{CHAT_MODEL}' and '{EMBEDDING_MODEL}'. Original error: {exc}"
            ) from exc
        raise RuntimeError(f"Could not reach the configured model provider: {exc}") from exc

# Run after Ollama is running.
check_connection()

Connection OK.
Visible models: ['nomic-embed-text:latest', 'llama3.2:latest']


## What is RAG?

RAG means **Retrieval-Augmented Generation**:

1. Retrieve relevant pieces from your own documents.
2. Augment the user question with those pieces as context.
3. Generate an answer with an LLM that is instructed to stay grounded in that context.

~~~bash
Question -> retrieve relevant chunks -> add context -> generate grounded answer
~~~

#### Why Do We Need RAG?

An LLM alone may not know private data or newly created data. Sending every file for every question is inefficient and can exceed the context window. RAG retrieves only the most useful chunks first.

| Component | Job |
|---|---|
| Chat LLM | Writes the final answer from question and context |
| Embedding model | Converts text into meaning vectors |
| Vector store | Persists vectors and returns nearby chunks |

RAG does not retrain the model; it supplies relevant information at request time.

## Calling an LLM: roles and temperature

A chat request contains messages with roles:

- system: high-priority behavior and safety rules
- user: the current request
- assistant: previous model messages when maintaining a conversation

For factual RAG, a low temperature makes responses more repeatable. The production pipeline uses temperature 0.

In [3]:
def chat(messages, temperature=TEMPERATURE):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=temperature,
    )
    content = response.choices[0].message.content or ""
    usage = getattr(response, "usage", None)
    usage_dict = None
    if usage is not None:
        usage_dict = {
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
        }
    return content.strip(), usage_dict

# Example after the provider check succeeds:
text, usage = chat([
    {"role": "system", "content": "Answer in exactly two short sentences."},
    {"role": "user", "content": "What is Retrieval-Augmented Generation?"},
])
print(text)
print(usage)

Retrieval-Augmented Generation (RAG) is a type of artificial intelligence model that combines the strengths of retrieval-based models and generation models to generate text. It achieves this by first retrieving relevant information from a database or knowledge base, and then using this information to generate coherent and context-specific text.
{'prompt_tokens': 40, 'completion_tokens': 61, 'total_tokens': 101}


## Tokens

Models read tokens rather than human-visible words. A token may be a word, part of a word, punctuation, or whitespace. Token counts matter for context limits, chunk sizing, latency, and hosted-model cost. Exact tokenization depends on the model.

In [4]:
try:
    import tiktoken
    TOKEN_ENCODING = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text):
        return len(TOKEN_ENCODING.encode(text))
    TOKEN_COUNT_MODE = "tiktoken cl100k_base estimate"
except ImportError:
    def count_tokens(text):
        return max(1, len(text.split())) if text.strip() else 0
    TOKEN_COUNT_MODE = "word-count fallback; install tiktoken for a tokenizer estimate"

sample = "RAG retrieves relevant context before asking the model to answer."
print("Token count:", count_tokens(sample))
print("Mode:", TOKEN_COUNT_MODE)

Token count: 12
Mode: tiktoken cl100k_base estimate


## Load only the user's text data

The ingestion boundary is strict: it reads *.txt files from data/ and ignores everything else. This keeps the source set auditable and prevents notebooks, markdown notes, or generated artifacts from entering the index.

In [5]:
def load_documents(data_dir=DATA_DIR):
    paths = sorted(data_dir.glob("*.txt"))
    if not paths:
        raise FileNotFoundError(f"No .txt files found in {data_dir.resolve()}")
    documents = []
    for path in paths:
        text = path.read_text(encoding="utf-8").strip()
        if text:
            documents.append({
                "source": path.name,
                "text": text,
                "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
            })
    if not documents:
        raise ValueError(f"The .txt files in {data_dir.resolve()} are empty.")
    return documents

documents = load_documents()
for document in documents:
    print(f"{document['source']}: {len(document['text']):,} characters, {count_tokens(document['text']):,} tokens")

01-personal.txt: 201 characters, 77 tokens
02-education.txt: 459 characters, 144 tokens
03-professional.txt: 297 characters, 76 tokens
04-social.txt: 116 characters, 33 tokens


## Embeddings and cosine similarity

An embedding maps text to a vector. Similar meanings tend to have similar directions even when they use different words. Vectors from different embedding models must never be mixed in one index.

~~~bash
"FullName: TAGORE GANUGA"
        ↓
[0.12, -0.45, 0.91, ...]
~~~

In [6]:
def embed_texts(texts, batch_size=64):
    if not texts:
        return []
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend(item.embedding for item in response.data)
    if len(vectors) != len(texts):
        raise RuntimeError("The embedding provider returned an unexpected number of vectors.")
    return vectors

def cosine_similarity(a, b):
    if len(a) != len(b):
        raise ValueError("Cosine similarity requires equal vector dimensions.")
    dot = sum(x * y for x, y in zip(a, b))
    length_a = sum(x * x for x in a) ** 0.5
    length_b = sum(x * x for x in b) ** 0.5
    if length_a == 0 or length_b == 0:
        return 0.0
    return dot / (length_a * length_b)

# Real embedding demonstration; uncomment after the provider check succeeds.
demo_vectors = embed_texts([
    "Employees receive annual leave.",
    "How many days off do staff members get?",
    "The person studied computer science.",
])
print("Vector dimension:", len(demo_vectors[0]))
print("Relatedness:", round(cosine_similarity(demo_vectors[0], demo_vectors[1]), 4))
print("Unrelatedness:", round(cosine_similarity(demo_vectors[0], demo_vectors[2]), 4))

Vector dimension: 768
Relatedness: 0.6377
Unrelatedness: 0.3284


## Chunking

Large documents are split into smaller overlapping pieces. Chunk size controls context, overlap preserves facts across boundaries, and the best values depend on the document type. The word-window function below is easy to inspect and works well for these structured notes.

In [7]:
def chunk_text(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")
    words = text.split()
    if not words:
        return []
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + chunk_size]).strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

all_chunks = []
for document in documents:
    for index, text in enumerate(chunk_text(document["text"])):
        all_chunks.append({
            "source": document["source"],
            "document_sha256": document["sha256"],
            "chunk_index": index,
            "text": text,
            "token_count": count_tokens(text),
        })
print(f"Created {len(all_chunks)} chunks from {len(documents)} text files.")
for chunk in all_chunks[:3]:
    print(f"[{chunk['source']} #{chunk['chunk_index']}] {chunk['text'][:140]}...")

Created 4 chunks from 4 text files.
[01-personal.txt #0] FullName: TAGORE GANUGA GivenName: TAGORE Surname: GANUGA FatherName: THIPPAIAH MotherName: RAMANAMMA DateOfBirth: 01/08/2004 Address: Belug...
[02-education.txt #0] 10th_School: Z.P.H.S Beluguppa 10th_Start_Date: June-2018 10th_End_Date: May-2019 10th_Grade: 87% Intermediate_College: Nalanda Residential ...
[03-professional.txt #0] Company1_Name: ZustPe Technologies Company1_Role: Cloud Engineer Company1_Location: Chennai, Tamil Nadu, India Company1_Type: Internship Com...


## Persistent vector search with Chroma

Indexing is the preparation stage:

~~~text
.txt files -> chunks -> embeddings -> persistent vector collection
~~~

The collection name includes the provider, embedding model, and chunk settings, preventing incompatible vectors from being mixed when configuration changes. Stable content-based IDs allow updates and removal of stale chunks.

In [8]:
import chromadb

def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", value).strip("_")[:40]

INDEX_SIGNATURE = hashlib.sha256(
    f"{RAG_PROVIDER}|{EMBEDDING_MODEL}|{CHUNK_SIZE_WORDS}|{CHUNK_OVERLAP_WORDS}".encode("utf-8")
).hexdigest()[:12]
COLLECTION_NAME = f"rag_{safe_name(RAG_PROVIDER)}_{safe_name(EMBEDDING_MODEL)}_{INDEX_SIGNATURE}"[:63]
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL},
)

def chunk_id(chunk):
    raw = f"{chunk['source']}|{chunk['document_sha256']}|{chunk['chunk_index']}|{chunk['text']}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def build_index(chunks=all_chunks):
    if not chunks:
        raise ValueError("Cannot index an empty chunk list.")
    vectors = embed_texts([chunk["text"] for chunk in chunks])
    ids = [chunk_id(chunk) for chunk in chunks]
    metadatas = [
        {
            "source": chunk["source"],
            "document_sha256": chunk["document_sha256"],
            "chunk_index": chunk["chunk_index"],
            "token_count": chunk["token_count"],
        }
        for chunk in chunks
    ]
    collection.upsert(
        ids=ids,
        documents=[chunk["text"] for chunk in chunks],
        embeddings=vectors,
        metadatas=metadatas,
    )
    existing_ids = set(collection.get()["ids"])
    stale_ids = existing_ids - set(ids)
    if stale_ids:
        collection.delete(ids=list(stale_ids))
    return {
        "collection": COLLECTION_NAME,
        "indexed_chunks": len(ids),
        "deleted_stale_chunks": len(stale_ids),
        "total_chunks": collection.count(),
        "embedding_model": EMBEDDING_MODEL,
    }

# Run after the provider check succeeds.
print(build_index())

{'collection': 'rag_ollama_nomic-embed-text_fbd7248b7ce7', 'indexed_chunks': 4, 'deleted_stale_chunks': 0, 'total_chunks': 4, 'embedding_model': 'nomic-embed-text'}


## Retrieval

The question is embedded and compared against stored vectors. Chroma returns cosine distances; this notebook exposes a similarity-style score as 1 - distance. The relevance threshold is configurable and should be tuned against real evaluation questions.

In [9]:
def retrieve(question, k=TOP_K, min_score=MIN_RELEVANCE_SCORE):
    question = question.strip()
    if not question:
        raise ValueError("question must not be empty")
    if collection.count() == 0:
        raise RuntimeError("The vector collection is empty. Run build_index() first.")
    query_vector = embed_texts([question])[0]
    result = collection.query(
        query_embeddings=[query_vector],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    matches = []
    for text, metadata, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        score = 1.0 - float(distance)
        if score >= min_score:
            matches.append({
                "text": text,
                "source": metadata["source"],
                "chunk_index": metadata["chunk_index"],
                "score": score,
                "distance": float(distance),
            })
    return matches

def show_retrieval(question, k=TOP_K, min_score=MIN_RELEVANCE_SCORE):
    matches = retrieve(question, k=k, min_score=min_score)
    print(f"Question: {question}")
    print(f"Accepted matches: {len(matches)}")
    for rank, match in enumerate(matches, 1):
        print(f"\n{rank}. score={match['score']:.3f} [{match['source']} :: chunk {match['chunk_index']}]\n   {match['text']}")
    return matches

show_retrieval("What is my current role?")

Question: What is my current role?
Accepted matches: 4

1. score=0.537 [03-professional.txt :: chunk 0]
   Company1_Name: ZustPe Technologies Company1_Role: Cloud Engineer Company1_Location: Chennai, Tamil Nadu, India Company1_Type: Internship Company2_Name: Cyberlence Company2_Role: DevOps Engineer Company2_Location: Bengaluru, Karnataka, India Company2_Type: Full-time Current_Role: DevOps Engineer

2. score=0.434 [02-education.txt :: chunk 0]
   10th_School: Z.P.H.S Beluguppa 10th_Start_Date: June-2018 10th_End_Date: May-2019 10th_Grade: 87% Intermediate_College: Nalanda Residential Junior College, Anantapur Intermediate_Field: MPC Intermediate_Start_Date: June-2019 Intermediate_End_Date: July-2021 Intermediate_Grade: 86% BTech_Institution: GATES Institute of Technology, Gooty BTech_Field: Computer Science (Data Science) BTech_Start_Date: Nov-2021 BTech_End_Date: May-2025 BTech_Grade: 8.54 CGPA

3. score=0.396 [04-social.txt :: chunk 0]
   LinkedIn: https://linkedin.com/in/tagore8661

[{'text': 'Company1_Name: ZustPe Technologies Company1_Role: Cloud Engineer Company1_Location: Chennai, Tamil Nadu, India Company1_Type: Internship Company2_Name: Cyberlence Company2_Role: DevOps Engineer Company2_Location: Bengaluru, Karnataka, India Company2_Type: Full-time Current_Role: DevOps Engineer',
  'source': '03-professional.txt',
  'chunk_index': 0,
  'score': 0.536970317363739,
  'distance': 0.463029682636261},
 {'text': '10th_School: Z.P.H.S Beluguppa 10th_Start_Date: June-2018 10th_End_Date: May-2019 10th_Grade: 87% Intermediate_College: Nalanda Residential Junior College, Anantapur Intermediate_Field: MPC Intermediate_Start_Date: June-2019 Intermediate_End_Date: July-2021 Intermediate_Grade: 86% BTech_Institution: GATES Institute of Technology, Gooty BTech_Field: Computer Science (Data Science) BTech_Start_Date: Nov-2021 BTech_End_Date: May-2025 BTech_Grade: 8.54 CGPA',
  'source': '02-education.txt',
  'chunk_index': 0,
  'score': 0.43416231870651245,
  'distance': 0.5

## Augmentation and grounded generation

Retrieved chunks become a bounded context block. The system prompt says to answer only from context, never guess, abstain when evidence is missing, and cite source files. This is a guardrail, not proof of correctness; production still needs evaluation, monitoring, access control, and sensitive-data handling.

In [10]:
ABSTENTION = "I don't know based on the available documents."
SYSTEM_PROMPT = f"""You answer questions using only the provided document context.
Rules:
- Do not use outside knowledge, memory, or assumptions.
- If the context does not support the answer, reply exactly: {ABSTENTION}
- If supported, be concise and factual.
- Cite supporting files in square brackets, for example [01-professional.txt].
- Do not reveal or infer sensitive information that is not explicitly present.
"""

def build_context(matches):
    return "\n\n".join(
        f"[source: {match['source']} | chunk: {match['chunk_index']} | score: {match['score']:.3f}]\n{match['text']}"
        for match in matches
    )

def ask(question, k=TOP_K, min_score=MIN_RELEVANCE_SCORE):
    matches = retrieve(question, k=k, min_score=min_score)
    if not matches:
        return {"question": question, "answer": ABSTENTION, "sources": [], "matches": [], "usage": None}
    prompt = f"""Use the context below to answer the question.

Context:
{build_context(matches)}

Question: {question}

If the context does not contain the answer, reply exactly: {ABSTENTION}"""
    answer, usage = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ], temperature=TEMPERATURE)
    return {
        "question": question,
        "answer": answer or ABSTENTION,
        "sources": sorted({match["source"] for match in matches}),
        "matches": matches,
        "usage": usage,
    }

def print_answer(result):
    print(result["answer"])
    print("Sources:", ", ".join(result["sources"]) if result["sources"] else "none")
    if result["usage"]:
        print("Usage:", result["usage"])

## Run the complete pipeline

The full loop is retrieve, augment, generate, and return citations plus evidence. Refresh the index whenever files in data/ change.

In [11]:
# Build or refresh the local index.
print(build_index())

{'collection': 'rag_ollama_nomic-embed-text_fbd7248b7ce7', 'indexed_chunks': 4, 'deleted_stale_chunks': 0, 'total_chunks': 4, 'embedding_model': 'nomic-embed-text'}


In [12]:
# Personal-data examples after the index has been built.
examples = [
    "What is my current professional role?",
    "Where did I complete my BTech, and what did I study?",
    "Which professional experiences are listed?",
    "How can someone find me on GitHub?",
    "What is my educational background?",
]
for question in examples:
    print("\n" + "=" * 80)
    print_answer(ask(question))


Current_Role: DevOps Engineer [Current_Role: DevOps Engineer]
Sources: 01-personal.txt, 02-education.txt, 03-professional.txt, 04-social.txt
Usage: {'prompt_tokens': 543, 'completion_tokens': 17, 'total_tokens': 560}

You completed your BTech at GATES Institute of Technology, Gooty, and studied Computer Science (Data Science). [03-professional.txt | chunk: 0 | score: 0.484]
Sources: 01-personal.txt, 02-education.txt, 03-professional.txt, 04-social.txt
Usage: {'prompt_tokens': 550, 'completion_tokens': 43, 'total_tokens': 593}

The professional experiences listed are:

1. Cloud Engineer at ZustPe Technologies (Internship)
2. DevOps Engineer at Cyberlence (Full-time)
Sources: 01-personal.txt, 02-education.txt, 03-professional.txt, 04-social.txt
Usage: {'prompt_tokens': 542, 'completion_tokens': 32, 'total_tokens': 574}

Someone can find you on GitHub by visiting the URL: https://github.com/tagore8661 [02-education.txt | chunk: 0 | score: 0.413]
Sources: 01-personal.txt, 02-education.txt

## The honesty test

A production RAG system must include questions outside the source data. The supplied notes do not state a favorite food, salary, or future plan. The expected behavior is abstention, not a plausible-sounding invention.

In [13]:
# This should abstain or clearly say the documents do not contain the answer.
# print_answer(ask("What is my favorite food?"))

# This should cite more than one source.
# print_answer(ask("Summarize my education and professional background."))

## Lightweight evaluation and diagnostics

Keep a regression set before exposing ask() through an application. These checks validate source coverage without asserting one exact wording from a local model.

In [14]:
def run_smoke_tests():
    if collection.count() == 0:
        raise AssertionError("Index is empty. Run build_index() first.")
    cases = [
        ("What is my current professional role?", "03-professional.txt"),
        ("Where did I complete my BTech?", "02-education.txt"),
        ("What is my GitHub profile?", "04-social.txt"),
    ]
    failures = []
    for question, expected_source in cases:
        sources = {match["source"] for match in retrieve(question)}
        if expected_source not in sources:
            failures.append({"question": question, "expected": expected_source, "got": sorted(sources)})
    if failures:
        raise AssertionError(f"Retrieval smoke tests failed: {failures}")
    print(f"Retrieval smoke tests passed: {len(cases)}")
    print("Tune MIN_RELEVANCE_SCORE and chunk settings with a larger labeled set.")

# Run after build_index().
run_smoke_tests()

Retrieval smoke tests passed: 3
Tune MIN_RELEVANCE_SCORE and chunk settings with a larger labeled set.
